# 02 - Silver to Gold

This notebook reads from the Silver schema and builds a star schema in the Gold schema.

**Dimensions:** dim_vehicle, dim_fuel_type, dim_location, dim_parking_area, dim_date  
**Facts:** fact_vehicle_registration, fact_vehicle_emissions, fact_fuel_distribution, fact_parking_capacity

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [ ]:
from pyspark.sql.functions import (
    col, monotonically_increasing_id, lit, sequence,
    explode, year, quarter, month, dayofmonth, dayofweek,
    date_format, weekofyear, when, min as spark_min, max as spark_max,
    expr, coalesce, to_date
)
from pyspark.sql.types import IntegerType, DateType
from datetime import date

print("Imports ready.")

## Dimension: dim_vehicle

In [ ]:
vehicles = spark.read.table("silver.vehicles")

dim_vehicle = vehicles.select(
    "kenteken", "voertuigsoort", "merk", "handelsbenaming",
    "inrichting", "eerste_kleur", "tweede_kleur",
    "europese_voertuigcategorie", "subcategorie_nederland",
    "aantal_zitplaatsen", "aantal_deuren", "aantal_wielen",
    "massa_ledig_voertuig", "massa_rijklaar",
    "lengte", "breedte", "hoogte_voertuig",
    "cilinderinhoud", "aantal_cilinders",
    "maximale_constructiesnelheid",
    "export_indicator", "taxi_indicator", "wam_verzekerd"
).dropDuplicates(["kenteken"]) \
 .withColumn("vehicle_key", monotonically_increasing_id())

dim_vehicle.write.mode("overwrite").saveAsTable("gold.dim_vehicle")
print(f"gold.dim_vehicle: {dim_vehicle.count():,} rows")

## Dimension: dim_fuel_type

In [ ]:
vehicle_fuels = spark.read.table("silver.vehicle_fuels")

dim_fuel_type = vehicle_fuels.select(
    "brandstof_omschrijving", "emissieklasse",
    "milieuklasse_eg_goedkeuring_licht", "milieuklasse_eg_goedkeuring_zwaar",
    "klasse_hybride_elektrisch_voertuig",
    "co2_emissieklasse", "uitlaatemissieniveau"
).dropDuplicates() \
 .withColumn("fuel_type_key", monotonically_increasing_id())

dim_fuel_type.write.mode("overwrite").saveAsTable("gold.dim_fuel_type")
print(f"gold.dim_fuel_type: {dim_fuel_type.count():,} rows")

## Dimension: dim_location

In [ ]:
# Postal codes from fuels_by_postal_code
fuels_pc4 = spark.read.table("silver.fuels_by_postal_code")
pc4_locations = fuels_pc4.select(
    col("postcode").cast("string").alias("postcode")
).distinct().withColumn("street_name", lit(None).cast("string")) \
 .withColumn("house_number", lit(None).cast("integer")) \
 .withColumn("zip_code", lit(None).cast("string")) \
 .withColumn("place", lit(None).cast("string")) \
 .withColumn("province", lit(None).cast("string")) \
 .withColumn("country", lit(None).cast("string"))

# Locations from parking addresses
parking_addr = spark.read.table("silver.parking_addresses")
park_locations = parking_addr.select(
    col("zipcode").substr(1, 4).alias("postcode"),
    col("streetname").alias("street_name"),
    col("housenumber").alias("house_number"),
    col("zipcode").alias("zip_code"),
    col("place"),
    col("province"),
    col("country")
).distinct()

# Union and deduplicate (parking addresses preferred for detail)
dim_location = park_locations.unionByName(pc4_locations) \
    .dropDuplicates(["postcode", "street_name", "house_number"]) \
    .withColumn("location_key", monotonically_increasing_id())

dim_location.write.mode("overwrite").saveAsTable("gold.dim_location")
print(f"gold.dim_location: {dim_location.count():,} rows")

## Dimension: dim_parking_area

In [ ]:
parking_specs = spark.read.table("silver.parking_area_specs")

dim_parking_area = parking_specs.select(
    "area_manager_id", "area_id",
    "capacity", "charging_point_capacity",
    "disabled_access", "maximum_vehicle_height", "limited_access"
).dropDuplicates(["area_manager_id", "area_id"]) \
 .withColumn("parking_area_key", monotonically_increasing_id())

dim_parking_area.write.mode("overwrite").saveAsTable("gold.dim_parking_area")
print(f"gold.dim_parking_area: {dim_parking_area.count():,} rows")

## Dimension: dim_date

In [ ]:
# Generate a date dimension from 1950-01-01 to 2030-12-31
date_range = spark.sql("""
    SELECT explode(sequence(
        to_date('1950-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )) AS full_date
""")

dim_date = date_range \
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("year", year(col("full_date"))) \
    .withColumn("quarter", quarter(col("full_date"))) \
    .withColumn("month", month(col("full_date"))) \
    .withColumn("month_name", date_format(col("full_date"), "MMMM")) \
    .withColumn("week_of_year", weekofyear(col("full_date"))) \
    .withColumn("day_of_month", dayofmonth(col("full_date"))) \
    .withColumn("day_of_week", dayofweek(col("full_date"))) \
    .withColumn("day_name", date_format(col("full_date"), "EEEE")) \
    .withColumn("is_weekend", when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False))

dim_date.write.mode("overwrite").saveAsTable("gold.dim_date")
print(f"gold.dim_date: {dim_date.count():,} rows")

## Fact: fact_vehicle_registration

In [ ]:
# Read dimension for joining
dim_v = spark.read.table("gold.dim_vehicle").select("kenteken", "vehicle_key")

# Build fact from silver.vehicles
fact_reg = vehicles.join(dim_v, "kenteken", "inner") \
    .withColumn("date_key_eerste_toelating",
        date_format(col("datum_eerste_toelating"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("date_key_tenaamstelling",
        date_format(col("datum_tenaamstelling"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("date_key_apk_vervaldatum",
        date_format(col("vervaldatum_apk"), "yyyyMMdd").cast(IntegerType())) \
    .select(
        "vehicle_key",
        "date_key_eerste_toelating", "date_key_tenaamstelling", "date_key_apk_vervaldatum",
        "bruto_bpm", "catalogusprijs", "massa_rijklaar", "massa_ledig_voertuig",
        "cilinderinhoud", "aantal_cilinders", "maximale_constructiesnelheid",
        "laadvermogen", "aantal_zitplaatsen", "aantal_deuren"
    )

fact_reg.write.mode("overwrite").saveAsTable("gold.fact_vehicle_registration")
print(f"gold.fact_vehicle_registration: {fact_reg.count():,} rows")

## Fact: fact_vehicle_emissions

In [ ]:
# Read dimensions for joining
dim_ft = spark.read.table("gold.dim_fuel_type")
fuel_type_join_cols = [
    "brandstof_omschrijving", "emissieklasse",
    "milieuklasse_eg_goedkeuring_licht", "milieuklasse_eg_goedkeuring_zwaar",
    "klasse_hybride_elektrisch_voertuig",
    "co2_emissieklasse", "uitlaatemissieniveau"
]

fact_emissions = vehicle_fuels \
    .join(dim_v, "kenteken", "inner") \
    .join(dim_ft, fuel_type_join_cols, "left") \
    .select(
        "vehicle_key", "fuel_type_key",
        "co2_uitstoot_gecombineerd", "co2_uitstoot_gewogen",
        "brandstofverbruik_stad", "brandstofverbruik_buiten_de_stad",
        "brandstofverbruik_gecombineerd",
        "nettomaximumvermogen", "roetuitstoot",
        "uitstoot_deeltjes_licht", "uitstoot_deeltjes_zwaar",
        "actieradius", "actieradius_extern_oplaadbaar",
        "emissie_co2_gecombineerd_wltp",
        "actie_radius_enkel_elektrisch_wltp"
    )

fact_emissions.write.mode("overwrite").saveAsTable("gold.fact_vehicle_emissions")
print(f"gold.fact_vehicle_emissions: {fact_emissions.count():,} rows")

## Fact: fact_fuel_distribution

In [ ]:
dim_loc = spark.read.table("gold.dim_location")

# Join fuels_by_postal_code to dim_location on postcode
fact_fuel_dist = fuels_pc4 \
    .withColumn("postcode_str", col("postcode").cast("string")) \
    .join(
        dim_loc.select("postcode", "location_key").dropDuplicates(["postcode"]),
        col("postcode_str") == dim_loc["postcode"],
        "left"
    ) \
    .select(
        "location_key",
        fuels_pc4["voertuigsoort"], fuels_pc4["brandstof"],
        fuels_pc4["extern_oplaadbaar"],
        fuels_pc4["aantal"]
    )

fact_fuel_dist.write.mode("overwrite").saveAsTable("gold.fact_fuel_distribution")
print(f"gold.fact_fuel_distribution: {fact_fuel_dist.count():,} rows")

## Fact: fact_parking_capacity

In [ ]:
# Join parking specs with parking addresses and date dim
dim_pa = spark.read.table("gold.dim_parking_area")

fact_parking = parking_specs \
    .join(dim_pa.select("area_manager_id", "area_id", "parking_area_key"),
          ["area_manager_id", "area_id"], "inner") \
    .withColumn("date_key_start",
        date_format(col("start_date"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("date_key_end",
        date_format(col("end_date"), "yyyyMMdd").cast(IntegerType())) \
    .select(
        "parking_area_key", "date_key_start", "date_key_end",
        "capacity", "charging_point_capacity", "disabled_access"
    )

fact_parking.write.mode("overwrite").saveAsTable("gold.fact_parking_capacity")
print(f"gold.fact_parking_capacity: {fact_parking.count():,} rows")

## Summary
All Gold tables have been created.

In [ ]:
gold_tables = [
    "dim_vehicle", "dim_fuel_type", "dim_location",
    "dim_parking_area", "dim_date",
    "fact_vehicle_registration", "fact_vehicle_emissions",
    "fact_fuel_distribution", "fact_parking_capacity"
]
print("\n=== GOLD LAYER SUMMARY ===")
for t in gold_tables:
    count = spark.read.table(f"gold.{t}").count()
    cols = len(spark.read.table(f"gold.{t}").columns)
    print(f"  gold.{t}: {count:,} rows, {cols} columns")
print("\nDone! Star schema is ready for semantic model.")